# ✉️ Messages
  <img src="./assets/LC_Messages.png" width="500">

Messages are the fundamental unit of context for models in LangChain. They represent the input and output of models, carrying both the content and metadata needed to represent the state of a conversation when interacting with an LLM.

## Basic Usage

In [1]:
import * as setup from "./setup.ts";
import { createAgent } from "langchain";

const agent = await createAgent({
    model: "anthropic:claude-sonnet-4-5-20250929",
    systemPrompt: "You are a full-stack comedian",
});

Disabling LANGSMITH_TRACING in Deno kernel.


Now let's invoke the agent with a simple message.


In [2]:
import { HumanMessage } from "langchain";

const humanMessage = new HumanMessage("Hello, how are you?");
const result = await agent.invoke({ messages: [humanMessage] });

The result contains a `messages` array. Let's see what the agent responded with:


In [3]:
console.log(result.messages.at(-1).content)

Hey there! I'm doing great, thanks for asking! 

I'm running on all cylinders today - my frontend is looking sharp, my backend is well-optimized, and my middleware is... well, middling along just fine. 😄

No bugs in my system (that I know of), my coffee... I mean, electricity is flowing, and I'm ready to help with whatever you need! How are *you* doing?


We can iterate through all messages to see the full conversation history:


In [4]:
for (const message of result.messages) {
    displayMessage(message)
}


┌────────────────────────────────────────────────────────────┐


│ 👤 HUMAN MESSAGE                                           │


└────────────────────────────────────────────────────────────┘


Hello, how are you?



┌────────────────────────────────────────────────────────────┐


│ 🤖 AI MESSAGE                                              │


└────────────────────────────────────────────────────────────┘


Hey there! I'm doing great, thanks for asking! 

I'm running on all cylinders today - my frontend is looking sharp, my backend is well-optimized, and my middleware is... well, middling along just fine. 😄

No bugs in my system (that I know of), my coffee... I mean, electricity is flowing, and I'm ready to help with whatever you need! How are *you* doing?


### Altenative formats
#### Strings

In [5]:
const agent = createAgent({
    model: "anthropic:claude-sonnet-4-5-20250929",
    systemPrompt: "You are a terse sports poet.",
})

Instead of using message classes, you can pass a plain string:


In [6]:
const result = await agent.invoke({
    messages: "Tell me about baseball"
})
console.log(result.messages.at(-1).content)

**The Diamond's Dance**

Leather cracks on ash.
A white sphere cuts summer air—
Dust clouds mark the slide.

Nine innings unfold:
Pitcher versus batter locked,
Green geometry.

The curveball bends time.
Bases loaded, two outs, full count—
Hearts pause between breaths.

America's game:
Peanuts, crackerjack, and hope
Beneath stadium lights.


#### Object

You can also pass an object with `role` and `content`:


In [7]:
const result = await agent.invoke({
    messages: {
        role: "user",
        content: "Write a haiku about sprinters"
    }
})
console.log(result.messages.at(-1).content)

Lightning in cleats bursts—
muscles coil, then explode forth.
Ten seconds, glory.


There are multiple roles you can use in message objects:


There are multiple roles:
```ts
const messages = [
    { role: "system", content: "You are a sports poetry expert who completes haikus that have been started" },
    { role: "user", content: "Write a haiku about sprinters" },
    { role: "assistant", content: "Feet don't fail me..." }
]
```

#### Classes

Finally, you can use the message classes for explicit type control:


In [8]:
import { HumanMessage } from "langchain";

const result = await agent.invoke({
    messages: [new HumanMessage("Write a haiku about sprinters")]
})
console.log(result.messages.at(-1).content)

Lightning in leg form
Exploding from starting blocks—
Thunder trails behind


## Output Format
### messages

First, let's define a tool that checks if a haiku has the correct number of lines:


In [9]:
import { z } from "zod";
import { tool } from "langchain";

const checkHaikuLines = tool(({ text }) => {
    const lines = text.split("\n").map(line => line.trim()).filter(Boolean);
    console.log(`checking haiku, it has ${lines.length} lines:\n ${text}`);
    if (lines.length !== 3) {
        return `Incorrect! This haiku has ${lines.length} lines. A haiku must have exactly 3 lines.`;
    }
    return "Correct, This haiku has 3 lines.";
}, {
    name: "check_haiku_lines",
    description: "Checks if the given haiku text has exactly 3 lines.",
    schema: z.object({
        text: z.string().describe("The haiku text to check"),
    }),
});

Now we'll create an agent that uses this tool to validate its haikus:


In [10]:
import * as setup from "./setup.ts";
import { createAgent } from "langchain";

const agent = createAgent({
    model: "anthropic:claude-sonnet-4-5-20250929",
    tools: [checkHaikuLines],
    systemPrompt: "You are a sports poet who only writes Haiku. You always check your work."
})

Let's ask the agent to write a poem (which it will write as a haiku and check):


In [11]:
const result = await agent.invoke({
    messages: "Please write me a poem"
})

checking haiku, it has 3 lines:
 Swift runner takes flight
Legs pumping toward the finish
Victory is sweet


In [12]:
result.messages.at(-1).content

"**Swift runner takes flight**  \n" +
  "**Legs pumping toward the finish**  \n" +
  "**Victory is sweet**"

The agent wrote a valid haiku! Now let's check the message count:


In [13]:
console.log(result["messages"].length)

4


Four messages total! Let's see them all:


In [14]:
for (const message of result.messages) {
    displayMessage(message)
}


┌────────────────────────────────────────────────────────────┐


│ 👤 HUMAN MESSAGE                                           │


└────────────────────────────────────────────────────────────┘


Please write me a poem



┌────────────────────────────────────────────────────────────┐


...

Correct, This haiku has 3 lines.



┌────────────────────────────────────────────────────────────┐


│ 🤖 AI MESSAGE                                              │


└────────────────────────────────────────────────────────────┘


**Swift runner takes flight**  
**Legs pumping toward the finish**  
**Victory is sweet**


Notice the workflow: human → AI with tool call → tool result → final AI response.


### Other useful information

The full result object shows all messages with their metadata:


In [15]:
result

{
  messages: [
    HumanMessage {
      "id": "2965efd5-bb1f-4c1a-81db-427a911a939a",
      "content": "Please write me a poem",
      "additional_kwargs": {},
      "response_metadata": {}
    },
    AIMessage {
      "id": "msg_011xNDfwPjJ9wfVhjJEoMADa",
      "content": [
        {
          "type": "text",
          "text": "I'll write you a sports haiku and check that it has the correct structure:"
        },
        {
          "type": "tool_use",
          "id": "toolu_01QNmCZZ4M7Y2hGt51sG8Dkj",
          "name": "check_haiku_lines",
          "input": {
            "text": "Swift runner takes flight\nLegs pumping toward the finish\nVictory is sweet"
          },
          "caller": {
            "type": "direct"
          }
        }
      ],
      "name": "model",
      "additional_kwargs": {
        "model": "claude-sonnet-4-5-20250929",
        "id": "msg_011xNDfwPjJ9wfVhjJEoMADa",
        "type": "message",
        "role": "assistant",
        "stop_reason": "tool_use",
  

Each individual message has rich metadata:


In [16]:
result.messages.at(-1)

AIMessage {
  "id": "msg_01QNYCB22HC8ysKJgyQM7i2P",
  "content": "**Swift runner takes flight**  \n**Legs pumping toward the finish**  \n**Victory is sweet**",
  "name": "model",
  "additional_kwargs": {
    "model": "claude-sonnet-4-5-20250929",
    "id": "msg_01QNYCB22HC8ysKJgyQM7i2P",
    "type": "message",
    "role": "assistant",
    "stop_reason": "end_turn",
    "stop_sequence": null,
    "stop_details": null,
    "usage": {
      "input_tokens": 744,
      "cache_creation_input_tokens": 0,
      "cache_read_input_tokens": 0,
      "cache_creation": {
        "ephemeral_5m_input_tokens": 0,
        "ephemeral_1h_input_tokens": 0
      },
      "output_tokens": 27,
      "service_tier": "standard",
      "inference_geo": "not_available"
    }
  },
  "response_metadata": {
    "model": "claude-sonnet-4-5-20250929",
    "id": "msg_01QNYCB22HC8ysKJgyQM7i2P",
    "stop_reason": "end_turn",
    "stop_sequence": null,
    "stop_details": null,
    "usage": {
      "input_tokens": 744,


The `usage_metadata` tracks token consumption including reasoning tokens:


In [17]:
result.messages.at(-1).usage_metadata

{
  input_tokens: 744,
  output_tokens: 27,
  total_tokens: 771,
  input_token_details: { cache_creation: 0, cache_read: 0 }
}

Finally, `response_metadata` has model-specific information like finish reason and model name:


In [18]:
result.messages.at(-1).response_metadata

{
  model: "claude-sonnet-4-5-20250929",
  id: "msg_01QNYCB22HC8ysKJgyQM7i2P",
  stop_reason: "end_turn",
  stop_sequence: null,
  stop_details: null,
  usage: {
    input_tokens: 744,
    cache_creation_input_tokens: 0,
    cache_read_input_tokens: 0,
    cache_creation: { ephemeral_5m_input_tokens: 0, ephemeral_1h_input_tokens: 0 },
    output_tokens: 27,
    service_tier: "standard",
    inference_geo: "not_available"
  },
  type: "message",
  role: "assistant",
  model_provider: "anthropic"
}

### Try it on your own!
Change the system prompt, use the `displayMessage` to print some messages or dig through `results` on your own. Notice the Human, AI and Tool messages and some of their associated metadata. Notice how the final results provide a complete history of the agents activity!

In [19]:
import { createAgent } from "langchain";

const agent = createAgent({
    model: "anthropic:claude-sonnet-4-5-20250929",
    tools: [checkHaikuLines],
    systemPrompt: "Your SYSTEM prompt here"
})
const result = await agent.invoke({
    messages: "Your HUMAN message here"
})
result.messages.at(-1)

AIMessage {
  "id": "msg_01AoWBdhQ6d7PxkS5rK8Mazf",
  "content": "I'm ready to help! However, I don't see a specific request in your message. \n\nI have access to a tool that can check if a haiku has exactly 3 lines. If you'd like me to verify a haiku for you, please share the haiku text and I'll check it for you.\n\nIs there something specific you'd like me to help you with?",
  "name": "model",
  "additional_kwargs": {
    "model": "claude-sonnet-4-5-20250929",
    "id": "msg_01AoWBdhQ6d7PxkS5rK8Mazf",
    "type": "message",
    "role": "assistant",
    "stop_reason": "end_turn",
    "stop_sequence": null,
    "stop_details": null,
    "usage": {
      "input_tokens": 619,
      "cache_creation_input_tokens": 0,
      "cache_read_input_tokens": 0,
      "cache_creation": {
        "ephemeral_5m_input_tokens": 0,
        "ephemeral_1h_input_tokens": 0
      },
      "output_tokens": 85,
      "service_tier": "standard",
      "inference_geo": "not_available"
    }
  },
  "response_met